# 📝 การบ้าน Day 3: Evaluation & Optimization
## Agentic RAG: From Zero to Hero

**กำหนดส่ง:** 1 สัปดาห์

## 🎓 กรอกข้อมูลนักศึกษา

In [ ]:
# ─── กรอกข้อมูลของคุณ ───
STUDENT_NAME = 'ภัทรภร พจนาพิมล'   # เช่น 'สมชาย ใจดี'
STUDENT_ID   = '673040627-5'   # เช่น '6512345678'
PHONE        = '0616010391'   # เช่น '081-234-5678'
LINE_ID      = 'sudo_podium'   # เช่น 'somchai.j'

# ─── ตรวจสอบ (ห้ามแก้ไข) ───
assert len(STUDENT_ID) >= 5, '❌ กรุณากรอกรหัสนักศึกษา!'
assert len(STUDENT_NAME) >= 2, '❌ กรุณากรอกชื่อ-นามสกุล!'

print(f'✅ ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'✅ รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

---
## 📦 ติดตั้ง Dependencies

In [ ]:
%%time
import importlib.util, subprocess, sys

def _pip_install(pkg_spec, import_name=None):
    pkg = pkg_spec.split('>=')[0].split('<=')[0].split('==')[0].split('[')[0].strip()
    imp = import_name or {
        'google-genai': 'google.genai', 'google-adk': 'google.adk',
        'sentence-transformers': 'sentence_transformers', 'qdrant-client': 'qdrant_client',
        'langchain-text-splitters': 'langchain_text_splitters',
        'langchain-huggingface': 'langchain_huggingface',
        'scikit-learn': 'sklearn', 'pymupdf': 'fitz',
        'docling-ibm-models': 'docling_ibm_models',
    }.get(pkg, pkg.replace('-', '_'))
    try:
        spec = importlib.util.find_spec(imp)
    except ModuleNotFoundError:
        spec = None
    has_version_constraint = any(op in pkg_spec for op in ('>=', '<=', '==', '>', '<', '!='))
    if spec is not None and not has_version_constraint:
        print(f'  \u23ed\ufe0f  {pkg}: skipped')
        return
    print(f'  \U0001f4e6 {pkg}: installing...', end='', flush=True)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg_spec],
                       capture_output=True, text=True)
    print(f'\r  \u2705 {pkg}: done' if r.returncode == 0 else f'\r  \u274c {pkg}: failed')
    if r.returncode != 0: print(r.stderr)

for _pkg in ['ragas', 'datasets', 'google-adk', 'google-genai', 'sentence-transformers', 'qdrant-client', 'langchain-text-splitters', 'rank_bm25']:
    _pip_install(_pkg)

import os, json, hashlib, random, asyncio, warnings
import numpy as np
warnings.filterwarnings('ignore')

try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
except:
    os.environ['GOOGLE_API_KEY'] = input('🔑 API Key: ')
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'False'

from google import genai
client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
print('✅ Setup complete')

## 📄 สร้างชุดข้อมูลเฉพาะตัว (ห้ามแก้ไข)

In [ ]:
%%time
# ─── Anti-Cheat: สร้างข้อมูลเฉพาะรหัสนักศึกษา ───
seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (10**9)
rng = random.Random(seed)

all_topics = {
    'healthcare': [
        'โรงพยาบาลศิริราชใช้ AI วิเคราะห์ภาพ X-ray ทรวงอก แม่นยำ 95% ลดเวลาวินิจฉัยจาก 30 นาทีเหลือ 5 นาที',
        'NLP วิเคราะห์เวชระเบียนอิเล็กทรอนิกส์ช่วยแพทย์สรุปประวัติผู้ป่วยได้เร็วขึ้น ลดเวลาจาก 15 นาทีเหลือ 2 นาที',
        'ระบบ AI ช่วยคัดกรองเบาหวานจากภาพถ่ายจอประสาทตา ใช้ Deep Learning ตรวจพบได้ตั้งแต่ระยะเริ่มต้น',
    ],
    'banking': [
        'ธนาคารกสิกรไทยใช้ RAG ตอบคำถามลูกค้า ลดภาระ Call Center 40% สามารถบริการ 24 ชั่วโมง',
        'ระบบตรวจจับการฉ้อโกงใช้ Machine Learning วิเคราะห์ธุรกรรมแบบ real-time ลดการฉ้อโกง 60%',
        'AI วิเคราะห์ความเสี่ยงสินเชื่อจากข้อมูลทางเลือก เช่น ประวัติค่าโทรศัพท์ ค่าน้ำค่าไฟ',
    ],
    'education': [
        'จุฬาลงกรณ์สร้างระบบ RAG ถาม-ตอบจากตำรา 500 เล่ม นักศึกษาเรียนรู้ด้วยตนเอง 24 ชั่วโมง',
        'Intelligent Tutoring System ปรับเนื้อหาตามระดับผู้เรียน ใช้ adaptive learning algorithm',
        'AI ช่วยตรวจข้อสอบอัตนัยภาษาไทย ลดเวลาตรวจ 70% ความแม่นยำ 88% เมื่อเทียบกับอาจารย์',
    ],
    'agriculture': [
        'Smart Farming ใช้ AI วิเคราะห์ภาพจากโดรน ตรวจโรคพืช 8 ชนิด ความแม่นยำ 92%',
        'ระบบพยากรณ์ผลผลิตข้าวจากข้อมูลดาวเทียม IoT sensor และสภาพอากาศ แม่นยำ 85%',
        'AI วิเคราะห์ราคาสินค้าเกษตรจากข่าวสารและ Social Media ช่วยเกษตรกรตัดสินใจการขาย',
    ],
    'logistics': [
        'ระบบ AI เพิ่มประสิทธิภาพเส้นทางขนส่งลดต้นทุนน้ำมัน 25% ใช้ Graph Neural Network',
        'คลังสินค้าอัจฉริยะใช้หุ่นยนต์ AGV และ Computer Vision จัดเรียงสินค้าอัตโนมัติ',
        'Chatbot บริการลูกค้าขนส่งใช้ NLP ภาษาไทย ตอบสถานะพัสดุและราคาส่งแบบ real-time',
    ],
}

topics = rng.sample(list(all_topics.keys()), 4)
my_data = {}
for t in topics:
    my_data[t] = rng.sample(all_topics[t], 2)

print(f'📋 Topics ของคุณ ({STUDENT_ID}):')
for t, texts in my_data.items():
    print(f'  📌 {t}: {len(texts)} texts')
    for tx in texts:
        print(f'     → {tx[:50]}...')

---
## 🎯 ขั้นตอนที่ 1: สร้าง RAG Pipeline (3 คะแนน)

สร้าง pipeline จากข้อมูล `my_data`:
1. Chunk ด้วย RecursiveCharacterTextSplitter
2. Embed ด้วย multilingual-e5-large
3. Upsert เข้า Qdrant
4. สร้างฟังก์ชัน `search_qdrant(query)` + `rag_answer(question)`

In [ ]:
%%time
# ─── ขั้นตอนที่ 1: สร้าง RAG Pipeline ───
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter

embed_model = SentenceTransformer('intfloat/multilingual-e5-large')
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)

all_chunks, all_sources = [], []
for src, texts in my_data.items():
    for text in texts:
        for chunk in splitter.split_text(text):
            all_chunks.append(chunk)
            all_sources.append(src)

print(f'📦 จำนวน chunks ทั้งหมด: {len(all_chunks)}')

# e5 models ต้องการ prefix "passage: " ตอน index และ "query: " ตอนค้นหา
chunk_embeddings = embed_model.encode(
    [f'passage: {c}' for c in all_chunks],
    normalize_embeddings=True,
    show_progress_bar=False,
)

qdrant = QdrantClient(':memory:')
COLLECTION_NAME = 'hw3_collection'
qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=chunk_embeddings.shape[1],
        distance=models.Distance.COSINE,
    ),
)

qdrant.upsert(
    collection_name=COLLECTION_NAME,
    points=[
        models.PointStruct(
            id=i,
            vector=chunk_embeddings[i].tolist(),
            payload={'text': all_chunks[i], 'source': all_sources[i]},
        )
        for i in range(len(all_chunks))
    ],
)
print(f'✅ Upsert {len(all_chunks)} chunks เข้า Qdrant แล้ว')


def search_qdrant(query, top_k=3):
    query_vec = embed_model.encode(f'query: {query}', normalize_embeddings=True).tolist()
    hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vec,
        limit=top_k,
    ).points
    return [
        {'text': h.payload['text'], 'source': h.payload['source'], 'score': h.score}
        for h in hits
    ]


def rag_answer(question, top_k=3):
    contexts = search_qdrant(question, top_k=top_k)
    context_text = '\n'.join(f'- [{c["source"]}] {c["text"]}' for c in contexts)
    prompt = f'''ตอบคำถามโดยอ้างอิงจากข้อมูลที่ให้เท่านั้น หากไม่มีข้อมูลที่เกี่ยวข้อง ให้บอกว่าไม่พบข้อมูล

ข้อมูล:
{context_text}

คำถาม: {question}

คำตอบ:'''
    resp = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.2),
    )
    return resp.text, contexts


print('✅ search_qdrant และ rag_answer พร้อมใช้งาน')

---
## 🎯 ขั้นตอนที่ 2: วัดคุณภาพ RAG (2.5 คะแนน)

1. สร้าง eval questions อย่างน้อย 5 ข้อ (จากข้อมูลของตัวเอง)
2. วัดด้วย **LLM-as-Judge** ให้คะแนน 1-5
3. แสดงผลคะแนนเฉลี่ย

In [ ]:
%%time
# ─── ขั้นตอนที่ 2: วัดคุณภาพ ───

def llm_judge(question, answer):
    prompt = f'''ให้คะแนน 1-5:
Q: {question}
A: {answer}
ตอบ JSON: {{"score": 1-5, "reason": "..."}}'''
    resp = client.models.generate_content(model='gemini-2.5-flash', contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.1, response_mime_type='application/json'))
    return json.loads(resp.text)

# สร้างคำถาม eval จาก topics ของตัวเอง (my_data) + คำถามรวม
eval_questions = [f'{src} มีการนำ AI มาใช้อย่างไรบ้าง?' for src in my_data.keys()]
eval_questions += [
    'ยกตัวอย่างการใช้ Machine Learning ที่กล่าวถึงในข้อมูล',
    'มีการกล่าวถึงตัวเลขความแม่นยำ (accuracy) กี่เปอร์เซ็นต์บ้าง?',
]

scores = []
for q in eval_questions:
    ans, _ = rag_answer(q)
    verdict = llm_judge(q, ans)
    scores.append(verdict['score'])
    print(f"  {'⭐'*verdict['score']} {q[:30]}...")

baseline_avg = sum(scores) / len(scores)
print(f'Average: {baseline_avg:.1f}/5.0')

---
## 🎯 ขั้นตอนที่ 3: ปรับปรุง Pipeline (2.5 คะแนน)

1. ปรับปรุงอย่างน้อย **2 อย่าง** (เช่น chunk_size + prompt)
2. วัดผล **Before/After** ด้วย LLM-as-Judge
3. อธิบายว่า **ทำไม** ดีขึ้นหรือไม่ดีขึ้น

In [ ]:
%%time
# ─── ขั้นตอนที่ 3: ปรับปรุง ───
# baseline_avg คำนวณไว้แล้วในขั้นตอนที่ 2

# ปรับปรุง 1: เพิ่ม chunk_size (200→350, overlap 30→50) ให้แต่ละ chunk มีบริบทมากขึ้น
splitter_v2 = RecursiveCharacterTextSplitter(chunk_size=350, chunk_overlap=50)

all_chunks_v2, all_sources_v2 = [], []
for src, texts in my_data.items():
    for text in texts:
        for chunk in splitter_v2.split_text(text):
            all_chunks_v2.append(chunk)
            all_sources_v2.append(src)

chunk_embeddings_v2 = embed_model.encode(
    [f'passage: {c}' for c in all_chunks_v2],
    normalize_embeddings=True,
    show_progress_bar=False,
)

COLLECTION_NAME_V2 = 'hw3_collection_v2'
qdrant.create_collection(
    collection_name=COLLECTION_NAME_V2,
    vectors_config=models.VectorParams(size=chunk_embeddings_v2.shape[1], distance=models.Distance.COSINE),
)
qdrant.upsert(
    collection_name=COLLECTION_NAME_V2,
    points=[
        models.PointStruct(id=i, vector=chunk_embeddings_v2[i].tolist(),
                            payload={'text': all_chunks_v2[i], 'source': all_sources_v2[i]})
        for i in range(len(all_chunks_v2))
    ],
)

def search_qdrant_v2(query, top_k=3):
    query_vec = embed_model.encode(f'query: {query}', normalize_embeddings=True).tolist()
    hits = qdrant.query_points(collection_name=COLLECTION_NAME_V2, query=query_vec, limit=top_k).points
    return [{'text': h.payload['text'], 'source': h.payload['source'], 'score': h.score} for h in hits]

# ปรับปรุง 2: ปรับ prompt ให้ตอบกระชับ + อ้างอิงแหล่งที่มา
def rag_answer_v2(question, top_k=3):
    contexts = search_qdrant_v2(question, top_k=top_k)
    context_text = '\n'.join(f'- [{c["source"]}] {c["text"]}' for c in contexts)
    prompt = f'''ตอบคำถามให้กระชับ ตรงประเด็น ไม่เกิน 3 ประโยค โดยอ้างอิงจากข้อมูลที่ให้เท่านั้น
และระบุแหล่งที่มา (source) ท้ายคำตอบในวงเล็บ หากไม่มีข้อมูลที่เกี่ยวข้อง ให้บอกว่าไม่พบข้อมูล

ข้อมูล:
{context_text}

คำถาม: {question}

คำตอบ:'''
    resp = client.models.generate_content(
        model='gemini-2.5-flash', contents=prompt,
        config=genai.types.GenerateContentConfig(temperature=0.2),
    )
    return resp.text, contexts

improved_scores = []
for q in eval_questions:
    ans, _ = rag_answer_v2(q)
    verdict = llm_judge(q, ans)
    improved_scores.append(verdict['score'])

improved_avg = sum(improved_scores) / len(improved_scores)
print(f'Before: {baseline_avg:.1f} → After: {improved_avg:.1f}')
print('เหตุผล: chunk_size ที่ใหญ่ขึ้น (200→350) ทำให้แต่ละ chunk มีบริบทครบถ้วนขึ้น '
      'ลดการตัดประโยคขาดตอนกลางใจความ ส่วน prompt ที่ปรับให้ตอบกระชับและอ้างอิงแหล่งที่มา '
      'ช่วยให้คำตอบตรงประเด็นและตรวจสอบได้ง่ายขึ้น ซึ่งมักทำให้คะแนนจาก LLM-as-Judge ดีขึ้น '
      '(ผลจริงอาจแตกต่างกันไปตามข้อมูลและคำถามของแต่ละคน)')

---
## 🎯 ขั้นตอนที่ 4: สร้าง Agent + Test (2 คะแนน)

1. สร้าง Agent ด้วย Google ADK ที่ใช้ Tool ค้นหาจาก RAG
2. เขียน test cases อย่างน้อย **5 ข้อ**
3. วัด pass rate

In [ ]:
%%time
# ─── ขั้นตอนที่ 4: Agent + Testing ───
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner
from google.adk.sessions import InMemorySessionService
from google.genai import types

def search_knowledge(query: str) -> dict:
    """ค้นหาข้อมูลที่เกี่ยวข้องจากฐานความรู้ (Qdrant) ด้วยคำค้นหา query"""
    results = search_qdrant(query, top_k=3)
    return {'results': results}

my_agent = Agent(
    name='hw3',
    model='gemini-2.5-flash',
    instruction=(
        'คุณเป็นผู้ช่วยตอบคำถามเกี่ยวกับการใช้ AI ในอุตสาหกรรมต่างๆ '
        'ใช้ tool search_knowledge เพื่อค้นหาข้อมูลก่อนตอบคำถามที่เกี่ยวกับข้อมูลเฉพาะเจาะจง '
        'หากคำถามเป็นการทักทายทั่วไปหรือไม่เกี่ยวข้องกับข้อมูล ไม่ต้องเรียก tool'
    ),
    tools=[search_knowledge],
)

runner = InMemoryRunner(agent=my_agent)
session_service = InMemorySessionService()

test_cases = [
    {'input': 'โรงพยาบาลใช้ AI ทำอะไรบ้าง?', 'expected_tool': 'search_knowledge'},
    {'input': 'ธนาคารใช้ AI ตรวจจับการฉ้อโกงอย่างไร?', 'expected_tool': 'search_knowledge'},
    {'input': 'สวัสดีครับ คุณคือใคร?', 'expected_tool': None},
    {'input': 'AI ช่วยเกษตรกรอย่างไรบ้าง?', 'expected_tool': 'search_knowledge'},
    {'input': 'วันนี้อากาศเป็นอย่างไร?', 'expected_tool': None},
]

async def run_test(test_case, user_id='test_user'):
    session = await session_service.create_session(app_name='hw3', user_id=user_id)
    called_tools = []
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=types.Content(role='user', parts=[types.Part(text=test_case['input'])]),
    ):
        if event.get_function_calls():
            for fc in event.get_function_calls():
                called_tools.append(fc.name)
    tool_called = called_tools[0] if called_tools else None
    passed = tool_called == test_case['expected_tool']
    return passed, tool_called

results = []
for tc in test_cases:
    passed, tool_called = await run_test(tc)
    results.append(passed)
    status = '✅' if passed else '❌'
    print(f"{status} '{tc['input'][:30]}...' → tool: {tool_called}")

pass_rate = sum(results) / len(results) * 100
print(f'\nPass rate: {pass_rate:.0f}% ({sum(results)}/{len(results)})')

---
## 📊 เกณฑ์การให้คะแนน

| ขั้นตอน | คะแนน | เกณฑ์ |
|---------|:-----:|------|
| 1. RAG Pipeline | 3 | Pipeline ทำงาน, ค้นหาได้, rag_answer ตอบได้ |
| 2. วัดคุณภาพ | 2.5 | eval questions ≥5, มีคะแนน LLM-as-Judge |
| 3. ปรับปรุง | 2.5 | ปรับ ≥2 อย่าง, มี Before/After, อธิบายเหตุผล |
| 4. Agent + Test | 2 | Agent ทำงาน, test cases ≥5, มี pass rate |
| **รวม** | **10** | |

## ✅ ตรวจสอบคำตอบ

In [ ]:
# ─── ตรวจสอบก่อนส่ง (ห้ามแก้ไข) ───
print('═' * 50)
print(f'📋 สรุปการบ้าน Day 3')
print(f'═' * 50)
print(f'ชื่อ: {STUDENT_NAME}')
print(f'รหัส: {STUDENT_ID}')
print(f'Topics: {list(my_data.keys())}')

checks = {
    'RAG Pipeline': 'search_qdrant' in dir() or 'search_qdrant' in globals(),
    'rag_answer': 'rag_answer' in dir() or 'rag_answer' in globals(),
    'llm_judge': 'llm_judge' in dir() or 'llm_judge' in globals(),
}

for name, ok in checks.items():
    print(f"  {'✅' if ok else '❌'} {name}")

if all(checks.values()):
    print('\n🎉 พร้อมส่ง!')
else:
    print('\n⚠️ ยังไม่ครบ กรุณาทำส่วนที่ขาด')